In [1]:
# ============================================================
# ORZYN AI m2.0
# Notebook 07
# Developer Intelligence
# ============================================================

from __future__ import annotations

In [2]:
from pathlib import Path
import sys

BACKEND_DIR = Path.cwd().parent

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

In [3]:
from collections import Counter
from dataclasses import dataclass

import pandas as pd

from orzyn import(
    client,
    get_active_repository,
)

In [4]:
repo_config = get_active_repository()

OWNER = repo_config.owner

REPOSITORY = repo_config.repository

print(f"Repository : {OWNER}/{REPOSITORY}")

Repository : facebook/react


In [5]:
CONTRIBUTORS_QUERY = """
query(
    $owner:String!,
    $name:String!
){

repository(

    owner:$owner,

    name:$name

){

mentionableUsers(

    first:100

){

nodes{

login

name

company

location

bio

url

avatarUrl

followers{

totalCount

}

repositories{

totalCount

}

}

}

}

}
"""

In [6]:
@dataclass(slots=True)
class DeveloperProfile:

    username: str

    name: str

    company: str | None

    location: str | None

    bio: str | None

    profile_url: str

    avatar_url: str

    followers: int

    repositories: int

In [7]:
def build_developer(node: dict) -> DeveloperProfile:

    return DeveloperProfile(

        username=node["login"],

        name=node["name"] or "",

        company=node["company"],

        location=node["location"],

        bio=node["bio"],

        profile_url=node["url"],

        avatar_url=node["avatarUrl"],

        followers=node["followers"]["totalCount"],

        repositories=node["repositories"]["totalCount"]

    )

In [8]:
result = client.execute(

    CONTRIBUTORS_QUERY,

    {

        "owner": OWNER,

        "name": REPOSITORY

    }

)

In [9]:
nodes = (

    result["repository"]

          ["mentionableUsers"]

          ["nodes"]

)

developers = [

    build_developer(node)

    for node in nodes

]

print(

    f"Downloaded {len(developers)} developer profiles."

)

Downloaded 100 developer profiles.


In [10]:
if not developers:

    raise RuntimeError(

        "No developers found."

    )

print("Developer data validated.")

Developer data validated.


In [11]:
top_developers = sorted(

    developers,

    key=lambda developer:

        developer.followers,

    reverse=True

)

top_developer = top_developers[0]

In [12]:
developers_df = pd.DataFrame(

    [

        {

            "Username": developer.username,

            "Name": developer.name,

            "Followers": developer.followers,

            "Repositories": developer.repositories,

            "Company": developer.company,

            "Location": developer.location,

            "Profile": developer.profile_url

        }

        for developer in top_developers

    ]

)

developers_df.head(100)

,Username,Name,Followers,Repositories,Company,Location,Profile
0,bvaughn,Brian Vaughn,11652,182,Citadel,"New York, NY",https://github.com/bvaughn
1,sophiebits,Sophie Alpert,10465,194,NaN,"San Francisco, CA",https://github.com/sophiebits
2,ljharb,Jordan Harband,8187,531,@socketdev @tc39,"Hillsborough, CA",https://github.com/ljharb
3,orta,Orta Therox,6255,1068,NaN,London / Huddersfield / NYC / Dublin / RDJ,https://github.com/orta
4,jdalton,John-David Dalton,5582,51,Socket.dev,Remote,https://github.com/jdalton
...,...,...,...,...,...,...,...
95,ludofischer,Ludovico Fischer,41,73,NaN,NaN,https://github.com/ludofischer
96,bspaulding,Bradley Spaulding,40,224,NaN,NaN,https://github.com/bspaulding
97,formido,Michael Terry,39,47,NaN,"Ventura, CA",https://github.com/formido
98,hekar,Hekar Khani,23,3,NaN,Canada,https://github.com/hekar


In [13]:
print("=" * 60)

print("Developer Summary")

print("=" * 60)

print(f"Profiles          : {len(developers)}")

print(f"Most Followed     : {top_developer.username}")

print(f"Followers         : {top_developer.followers}")

print(f"Average Followers : {developers_df['Followers'].mean():.2f}")

print(f"Average Repos     : {developers_df['Repositories'].mean():.2f}")

Developer Summary
Profiles          : 100
Most Followed     : bvaughn
Followers         : 11652
Average Followers : 1100.00
Average Repos     : 178.04
